# Task 3: Resume / Candidate Screening System
**Future Interns ML Internship 2026**  
**Intern:** Akalya A | **CIN:** FIT/MAR26/ML6268

## Objective
Build an ML system that automatically screens, scores, and ranks resumes based on a job description — extracting skills, scoring relevance, and identifying skill gaps.

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries loaded!')

## Step 1: Define Job Description & Sample Resumes

In [ ]:
# Job Description for a Machine Learning Engineer role
job_description = """
We are looking for a Machine Learning Engineer with strong Python skills.
Required skills: Python, scikit-learn, TensorFlow, deep learning, neural networks,
data preprocessing, feature engineering, model deployment, REST API, Flask, Git, SQL,
pandas, numpy, NLP, computer vision, data visualization, matplotlib, seaborn.
Experience with cloud platforms like AWS or GCP is a plus.
Good communication skills and ability to work in an agile team environment.
Bachelor's degree in Computer Science, Data Science or related field required.
"""

# Sample resumes (in real task, load from Kaggle dataset)
resumes = [
    {
        'name': 'Arun Kumar',
        'text': """
        Experienced ML Engineer with 3 years of experience. Proficient in Python, scikit-learn,
        TensorFlow, deep learning, neural networks, NLP, computer vision, pandas, numpy.
        Built and deployed REST API using Flask on AWS. Strong Git and SQL skills.
        Data preprocessing, feature engineering, model deployment experience.
        B.Tech Computer Science, 8.5 CGPA. Good communication skills, agile methodology.
        """
    },
    {
        'name': 'Priya Sharma',
        'text': """
        Data Analyst with Python and SQL expertise. Experience with pandas, numpy, matplotlib,
        seaborn, data visualization. Some machine learning using scikit-learn for classification.
        Basic knowledge of Git. B.Sc Statistics. Completed online courses in data science.
        Limited experience with deep learning or cloud platforms.
        """
    },
    {
        'name': 'Ravi Shankar',
        'text': """
        Backend Developer specializing in Java and Spring Boot. REST API development,
        SQL databases, Git version control. Some Python scripting knowledge.
        No experience with machine learning, TensorFlow, or data science tools.
        B.Tech Computer Science. Good team player with agile experience.
        """
    },
    {
        'name': 'Meena Devi',
        'text': """
        AI/ML Researcher with expertise in deep learning, neural networks, NLP, computer vision.
        Python, TensorFlow, PyTorch, scikit-learn, pandas, numpy proficiency.
        Published papers in NLP and computer vision. Model deployment on AWS and GCP.
        Flask REST API development. Feature engineering, data preprocessing expert.
        M.Tech AI, Git, SQL, matplotlib, seaborn. Excellent communication. Agile projects.
        """
    },
    {
        'name': 'Karthik Raj',
        'text': """
        Fresh graduate with B.Tech Computer Science. Completed projects in Python, machine learning
        using scikit-learn, pandas, numpy, matplotlib. Basic deep learning knowledge.
        SQL databases, Git. Coursera ML certificate. Eager to learn TensorFlow, cloud platforms.
        Good communication skills. Seeking entry-level ML role.
        """
    }
]

print(f'Job role: Machine Learning Engineer')
print(f'Number of candidates: {len(resumes)}')

## Step 2: Define Skills Dictionary

In [ ]:
# Required skills for the role
required_skills = [
    'python', 'scikit-learn', 'sklearn', 'tensorflow', 'deep learning', 'neural networks',
    'data preprocessing', 'feature engineering', 'model deployment', 'rest api',
    'flask', 'git', 'sql', 'pandas', 'numpy', 'nlp', 'computer vision',
    'data visualization', 'matplotlib', 'seaborn', 'aws', 'gcp',
    'communication', 'agile', 'machine learning'
]

def extract_skills(text, skills_list):
    """Extract which skills from the list appear in the text."""
    text_lower = text.lower()
    found = [skill for skill in skills_list if skill in text_lower]
    return found

# Extract skills from job description
jd_skills = extract_skills(job_description, required_skills)
print(f'Skills in job description ({len(jd_skills)}):')
print(jd_skills)

## Step 3: Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

jd_clean = preprocess(job_description)
resume_texts = [r['text'] for r in resumes]
resume_clean = [preprocess(t) for t in resume_texts]
print('Text preprocessing complete.')

## Step 4: Cosine Similarity Scoring

In [ ]:
# TF-IDF vectorize job description + all resumes together
all_docs = [jd_clean] + resume_clean
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(all_docs)

# JD vector is index 0, resumes are 1 onwards
jd_vector = tfidf_matrix[0]
resume_vectors = tfidf_matrix[1:]

# Cosine similarity between JD and each resume
similarities = cosine_similarity(jd_vector, resume_vectors).flatten()

# Skill-based score: what % of required skills does each resume have?
skill_scores = []
matched_skills = []
missing_skills = []

for r in resumes:
    found = extract_skills(r['text'], jd_skills)
    missing = [s for s in jd_skills if s not in found]
    skill_pct = len(found) / len(jd_skills) if jd_skills else 0
    skill_scores.append(skill_pct)
    matched_skills.append(found)
    missing_skills.append(missing)

# Combined score: 60% TF-IDF similarity + 40% skill coverage
final_scores = 0.6 * similarities + 0.4 * np.array(skill_scores)
final_scores_pct = (final_scores / final_scores.max() * 100).round(1)

# Build results dataframe
results = pd.DataFrame({
    'Candidate': [r['name'] for r in resumes],
    'Similarity_Score': (similarities * 100).round(1),
    'Skill_Coverage': (np.array(skill_scores) * 100).round(1),
    'Final_Score': final_scores_pct,
    'Skills_Matched': [len(m) for m in matched_skills],
    'Skills_Missing': [len(m) for m in missing_skills]
})

results = results.sort_values('Final_Score', ascending=False).reset_index(drop=True)
results.index += 1
results.index.name = 'Rank'

print('CANDIDATE RANKING')
print('='*60)
print(results.to_string())

## Step 5: Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Resume Screening Dashboard', fontsize=15, fontweight='bold')

# Plot 1: Candidate ranking bar chart
colors = ['#2ecc71' if i == 0 else '#3498db' if i == 1 else '#95a5a6'
          for i in range(len(results))]
bars = axes[0].barh(results['Candidate'][::-1], results['Final_Score'][::-1],
                    color=colors[::-1], edgecolor='black', alpha=0.85)
axes[0].set_xlabel('Final Score (out of 100)')
axes[0].set_title('Candidate Rankings — Final Score')
axes[0].set_xlim(0, 115)
for bar, val in zip(bars, results['Final_Score'][::-1]):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontweight='bold')

# Plot 2: Skill coverage heatmap
skill_matrix = []
for r in resumes:
    row = [1 if s in r['text'].lower() else 0 for s in jd_skills[:10]]
    skill_matrix.append(row)

skill_df = pd.DataFrame(skill_matrix,
                        index=[r['name'] for r in resumes],
                        columns=[s.capitalize() for s in jd_skills[:10]])
# Reorder by final score
skill_df = skill_df.reindex(results['Candidate'].tolist())

sns.heatmap(skill_df, annot=True, fmt='d', cmap='YlGn',
            linewidths=0.5, ax=axes[1], cbar=False)
axes[1].set_title('Skill Coverage Heatmap\n(1 = Present, 0 = Missing)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('task3_resume_screening.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as task3_resume_screening.png')

## Step 6: Detailed Candidate Report

In [ ]:
print('DETAILED CANDIDATE REPORTS')
print('='*60)

for idx, row in results.iterrows():
    name = row['Candidate']
    i = [r['name'] for r in resumes].index(name)
    print(f'\nRank #{idx} — {name}')
    print(f'  Final Score    : {row["Final_Score"]}/100')
    print(f'  Skill Coverage : {row["Skill_Coverage"]}% ({row["Skills_Matched"]}/{len(jd_skills)} skills)')
    print(f'  Text Similarity: {row["Similarity_Score"]}%')

    verdict = 'SHORTLIST' if row['Final_Score'] >= 60 else 'MAYBE' if row['Final_Score'] >= 40 else 'REJECT'
    print(f'  Verdict        : {verdict}')

    if missing_skills[i]:
        print(f'  Missing Skills : {", ".join(missing_skills[i][:5])}')
    else:
        print('  Missing Skills : None — perfect fit!')

print('\n' + '='*60)
shortlisted = results[results['Final_Score'] >= 60]
print(f'\nRecommendation: Interview the top {len(shortlisted)} candidate(s):')
for _, r in shortlisted.iterrows():
    print(f'  - {r["Candidate"]} (Score: {r["Final_Score"]})')

## Conclusion
This notebook built a complete ML-based resume screening system:
- Parsed and preprocessed resume text using NLTK
- Extracted required skills from job descriptions
- Scored resumes using TF-IDF cosine similarity
- Combined skill coverage + text similarity into a final weighted score
- Ranked candidates and identified skill gaps
- Visualized results with ranking charts and skill heatmaps

**Business Impact:** Reduces manual resume review from hours to seconds. Provides transparent, explainable scores. Identifies skill gaps for targeted interviews.

**Dataset:** Resume Dataset (Kaggle) / Custom resumes  
**Libraries:** pandas, nltk, scikit-learn, matplotlib, seaborn